# Spectral Data prep

Whether conducting statistical analyses or analysing separability, raw spectral refectance data should be prepared before ingesting into the `spectral_separability.ipynb` or `spectral_stats.ipynb`. This notebook walks through the various data preparation steps available in this repo.

1. Data Labelling - make new labels based on existing classifications
2. Cropping noisy wavelengths - cut down spectral range extremities to highest SNR region
3. Normalize data - Normalize all data to the range 0-1 (MinMax) or 0 mean and 1 st.dev (Standard Scaler)
4. Filter to targets - Reduce sample set to targets of interest
5. Denoise spectra - Use wavelet decomposition to remove high frequency noise
6. Resample spectra - Use Relative Spectral Responses from locally stored files to simulate other sensors

In [1]:
from random import sample
import pandas as pd
import matplotlib.pyplot as plt
import auxiliary.preprocessing as prep
import auxiliary.hyper_denoiser as hyper_denoiser
import auxiliary.resampling as resamp
import auxiliary.tools as tools

In [2]:
def scale_down_overexposed_spectra(data, labels, plot = True, ref_class = "durvillaea"):

    """
    Scale down the reflectance values for spectra that were previously overexposed (i.e. reflectance values above 1) by matching each over exposed spectrum with a spectrum from that same class in the proper value range.
    Where all spectra of a class are overexposed, the reference spectrum will be drawn from 'durvillaea' (or another defined reference class) as this is the most spectrally diverse class.
    
    Args
    data (pd.DataFrame): DataFrame containing the spectral data, with spectra as rows and wavelengths as columns.
    labels (pd.DataFrame): DataFrame containing the labels for each spectrum.
    ref_class = str: The class to use as a reference for scaling when all spectra of a class are overexposed. Default is "durvillaea".
    
    Outputs    Adjusted DataFrame with overexposed spectra scaled down to match the maximum reflectance values of good spectra from the same class, or from the reference class if all spectra of that class are overexposed.
    
    """
    
    over = data.loc[(data.max(axis = 1) >1), :]
    over_labels = labels.loc[over.index]
    good = data.loc[(data.max(axis = 1) <1), :]
    good_labels = labels.loc[good.index]
    
    for class_ in over_labels["Class"].unique():
        if class_ not in good_labels["Class"].unique():
            print(f"All spectra of class {class_} are overexposed. Using {ref_class} as reference for scaling.")
            over_labels.loc[over_labels["Class"] == class_, "Class"] = ref_class

    adjusted = data.copy()
    # bad_labels.replace("grass", "durvillaea", inplace= True)
    for spectrum in over.index:
        over_class = over_labels.loc[spectrum, "Class"]
        good_example = sample(good[good_labels["Class"] == over_class].index.tolist(), k = 1)
        proper_max = good.loc[good_example, :].max(axis = 1).values[0]
        adjusted.loc[spectrum, :] = over.loc[spectrum, :]/over.loc[spectrum, :].max()*proper_max
    
    if plot: 
        plt.plot(over.T)
        plt.title("Original overexposed spectra")
        plt.show()
        
        plt.plot(adjusted.loc[over.index, :].T)
        plt.title("Adjusted overexposed spectra")
        plt.show()
    
    return adjusted

In [3]:
# Define preprocessing parameters for hyperspectral data
full_res_path = "data/raw/combined_spectra.csv" #mine_red_green_spectra.csv"
bad_bands_list = list(map(str,(range(753, 769))))
bad_bands_treatment = "interpolate"
scaler = "standard"
target_scheme = "all"
target_sites = None
target_setup = None
start_nm = 460
end_nm = 925

In [4]:
# Initialize the preprocessing class (hyperspectral data only)
prepper = prep.PreprocessRefl(data_path = full_res_path,
                              bad_bands = bad_bands_list,
                              bad_bands_treatment= bad_bands_treatment,
                              scaler = scaler,
                              target_scheme = target_scheme,
                              target_setup = target_setup,
                              target_sites= target_sites,
                              start_nm = start_nm,
                              end_nm = end_nm)

In [ ]:
prepper.import_data()
prepper.make_new_labels()
prepper.spectra, prepper.labels = tools.sort_classes(prepper.spectra, prepper.labels,  labels_col = "Class")
prepper.filter_targets()
prepper.crop_noisy_wavelengths()

data = prepper.spectra_reduc
labels = prepper.labels_filtered

In [6]:
labels.to_csv("./data/feb2026_combo_labels_prepped.csv")

# Denoising

Before picking your level of noise to remove, it is smart to check out a few different levels to make sure the useful information in the signal is being preserved.
Use the plotting function below to compare noise removal levels, then calculate it for the whole dataset when you are satisfied.

In [ ]:
# Pick a sample to investigate and sigma adjustments to try (higher=more smoothing)
sample = 204

sigmas_levels = [0.5, 1, 1.5]
data = data.dropna(axis = 1)


# Plot the chosen samples denoised with the indicated sigma adjustments
fig, ax = plt.subplots(1,3)
ax[0].plot(data.iloc[sample, :].dropna(), color = "red")
ax[0].plot(pd.DataFrame(hyper_denoiser.denoise_spectrum(spectrum = data.iloc[sample, :], sigma_adjust=sigmas_levels[0])).set_index(data.columns), color = "blue")
ax[0].set_title(f"sigma_adjust = {sigmas_levels[0]}")

ax[1].plot(data.iloc[sample, :].dropna(), color = "red")
ax[1].plot(pd.DataFrame(hyper_denoiser.denoise_spectrum(spectrum = data.iloc[sample, :], sigma_adjust=sigmas_levels[1])).set_index(data.columns), color = "blue")
ax[1].set_title(f"sigma_adjust = {sigmas_levels[1]}")

ax[2].plot(data.iloc[sample, :].dropna(), color = "red")
ax[2].plot(pd.DataFrame(hyper_denoiser.denoise_spectrum(spectrum = data.iloc[sample, :], sigma_adjust=sigmas_levels[2])).set_index(data.columns), color = "blue")
ax[2].set_title(f"sigma_adjust = {sigmas_levels[2]}")

plt.show()

In [22]:
sigma_adjust = 0.5
quiet_data = pd.DataFrame(index = data.index, columns = data.columns)

for row in range(data.shape[0]):
    spectrum = data.iloc[row, :].dropna()
    quiet_data.iloc[row, :] = hyper_denoiser.denoise_spectrum(spectrum, sigma_adjust = sigma_adjust)

# Adjust for over-exposure

In some spectra taken in the intertidal zone or under changing conditions, the reflectance magnitude is unrealistic (i.e. over 1) but the shape is clear of signs of contamination. For these spectra to be used in pixel simulations, they must be corrected first. The correction below finds over-exposed spectra and samples the properly exposed (i.e. all reflectance values below 1) of that same class to determine an appropriate value to which the spectra should be scaled. the overexposed spectrum is then adjusted to have the same maximum reflectance value.

In [ ]:
scaled_data = scale_down_overexposed_spectra(data, labels, plot = False)

In [ ]:
scaled_data.head()

# Normalize 



In [5]:

# If you want to use de-noised data, uncomment the next line
# prepper.spectra_reduc = quiet_data
# prepper.spectra_reduc = scaled_data
prepper.normalize_data()
data = prepper.spectra_norm

# Export the full spectral resolution prepared data

In [6]:
#Adjust export name and location as desired
data.to_csv("feb2026_standard_noisy_prepped_data_fullres.csv", index = True)


## __________________________________________________________________________________________________________________________________________________________
##
# 📡 Resample to satellite sensors

Spectrally resample the dataset to match the relative spectral response (RSR) functions of satellite sensors. Can be implemented for a single sensor at a time or as a batch application to all available RSRs in a directory

In cases where there is only a small overlap between the destination RSR and the existing spectral data, the resampled output may be inaccurately low. To avoid this, use the `minimum_response_trim` function to limit to destination bands that have good coverage of your data. For example, setting the parameter to `0.1` will only output destination bands in which at least 10% of their relative spectral response is in the spectral range of your data.

In [ ]:
resamp.resample_to_all(rsr_dir = "satellite_RSRs/", spectra = scaled_data, out_dir= "data/processed/resampled/mine_and_specchio/sub-1-spectra_no-standardization/", minimum_response_trim= 0.1, prefix = "noisy")